# Prediction baseline research plan

**Scope:** Week-one evaluation of the community evolution prediction prototype.

**Current status:** The synthetic-data pipeline completed the table, temporal split, baseline prediction, and evaluation stages. The shared notebook still requires review and validation.

**Evidence run:** `lan-demo-final-20261020`

**Verified source revision:** `696cbb61da2faebb75f06a701afee943fe0199c0`

This notebook records the research design, evaluation evidence, limitations, and outstanding questions. Synthetic results must not be interpreted as real Reddit forecasting performance.

## Research question

Can the current prototype generate temporally valid predictions and evaluate community evolution events without using information unavailable at prediction time?

Week-one objectives:
- Verify the data-to-evaluation pipeline.
- Compare the available baseline models on the same temporal splits.
- Inspect class-level errors and potential leakage.
- Document what is still needed before evaluating real Reddit data.

The current experiment validates pipeline behavior on synthetic data. It does not establish real-world forecasting effectiveness or support conclusions about H3.

## Data and labels

The verified run uses synthetic community-month samples generated with seed 42.

- Total generated samples: 1,570.
- Labeled samples included in evaluation: 1,524.
- Unlabeled samples excluded from evaluation: 46.
- Event classes: stable, grow, shrink, merge, split, dissolve.

## Synthetic data and label generation

The current prototype uses synthetic community histories to test the forecasting pipeline before evaluating it on real Reddit data.

- **Dataset size:** 1,570 generated rows, including 1,524 labeled rows and 46 unlabeled rows.
- **Time span:** 36 months, from January 2015 to December 2017.
- **Random seed:** 42.
- **Target:** predict a community's next-month evolution label using information available up to the current month.
- **Feature timing:** features at time \(t\) must not use information from after time \(t\). The target corresponds to the subsequent month.
- **Unlabeled rows:** rows without a usable target label are not included in supervised evaluation.

The generator models community growth and evolution using synthetic histories, noise, and event-label rules. Its results are useful for testing the pipeline and evaluation logic, but they do not establish predictive performance on real Reddit communities.

Synthetic labels and features are not observations collected from Reddit. They must not be used to claim real-world performance.

## Split and leakage

The pipeline groups samples by label_month instead of randomly splitting community rows.

| Split | Samples | Labeled months |
|---|---:|---:|
| Train | 1,000 | 23 |
| Validation | 263 | 6 |
| Test | 261 | 6 |
| Unlabeled, excluded | 46 | 0 |

Required checks:
- Verify the exact first and last label_month in each split.
- Confirm that preprocessing and learned parameters use training data only.
- Verify that features are available at prediction time and targets refer to the intended future period.
- Run the negative-control leakage test and confirm it detects the deliberately leaky fixture.

These checks reduce known risks but do not prove that all possible leakage has been eliminated.

## Baselines and fairness

The verified evaluation includes:
- B1 Majority.
- B1 Persistence.
- B4 Logistic Regression.

All three models have metrics for train, validation, and test. Compare them on the same split and sample population.

## Baseline models and implementation

The prototype currently evaluates three baselines:

1. **B1 Majority:** predicts the most frequent class observed in the training data.
2. **B1 Persistence:** uses the previous evolution label as the prediction when that history is available.
3. **B4 Logistic Regression:** learns a classification model from community-level features.

For a fair comparison, models must use the same temporal train, validation, and test splits. Preprocessing and model fitting must use training data only. Validation and test data must not influence training or preprocessing decisions.

The implementation details—including the exact fallback behavior for missing history, feature list, model parameters, and preprocessing—should be checked against the source code before being reported as verified facts.

B2 Random Forest, B5 Gradient Boosting, HeTCo, and real Reddit features are not included in this verified run. No conclusion about their performance can be drawn.

## Results and evidence

The results cell below reads metrics and split counts from the verified run's artifacts rather than manually entering scores.

Primary metrics:
- macro_f1: F1 averaged over classes present in the true labels.
- macro_f1_six: F1 averaged over all six event classes.
- balanced_accuracy: mean recall over classes present in the true labels.

All six classes are present in the current train, validation, and test splits, so macro_f1 and macro_f1_six coincide in this run.

Interpret results alongside per-class metrics and confusion matrices. Report classes with no correct detections, not only aggregate scores.

All reported metrics must be traceable to the saved run artifacts. If the artifact, configuration, or source revision differs from the verified run, rerun the checks before treating the results as reproducible.

## Critical review

In the test split, B1 Persistence achieves macro-F1 of approximately 0.2557, B4 Logistic Regression approximately 0.2160, and B1 Majority approximately 0.1125.

Persistence performs best among the three currently evaluated models, but the scores remain limited. The confusion matrices show that some event classes are not detected correctly. Aggregate metrics alone therefore provide an incomplete picture.

The test period contains only six labeled months, and the experiment uses synthetic data. These results do not establish performance on real Reddit communities. Further review is needed for temporal boundaries, feature availability, class-level errors, and leakage controls.

## Implementation

Run the pipeline from the repository root using the project's virtual environment and explicit configuration:

```powershell
& .\.venv\Scripts\python.exe -m reddit_sna.pipelines.prediction `
  --config .\configs\prediction.toml `
  --run-id <unique-run-id>
```

For the verified run, inspect its manifest, split_summary.csv, metrics.csv, per_class.csv, and confusion matrices under artifacts/prediction/lan-demo-final-20261020/.

Do not modify sys.path to make imports work. Keep reusable production logic in src/reddit_sna rather than duplicating it in the notebook.

In [ ]:

import json
from pathlib import Path

import pandas as pd

# Locate the repository root without modifying sys.path.
current = Path.cwd().resolve()
repo_root = next(
    (
        parent
        for parent in (current, *current.parents)
        if (parent / "configs" / "prediction.toml").is_file()
        and (parent / "src" / "reddit_sna").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError(
        "Repository root not found. Open the notebook from "
        "the project workspace."
    )

run_id = "lan-demo-final-20261020"
run_dir = repo_root / "artifacts" / "prediction" / run_id

required_files = [
    "manifest.json",
    "split_summary.csv",
    "metrics.csv",
    "per_class.csv",
    "confusion_B1_majority_test.csv",
    "confusion_B1_persistence_test.csv",
    "confusion_B4_logreg_test.csv",
]

missing = [
    name for name in required_files
    if not (run_dir / name).is_file()
]
if missing:
    raise FileNotFoundError(
        f"Missing artifacts in {run_dir}: {missing}. "
        "Run the pipeline or select an existing run."
    )

manifest = json.loads(
    (run_dir / "manifest.json").read_text(encoding="utf-8")
)

assert manifest["status"] == "complete_through_baselines"

stage_status = manifest["stage_status"]
required_stages = ("table", "split", "baselines", "evaluation")
assert all(
    stage_status.get(stage) == "complete"
    for stage in required_stages
)

print("Run:", run_id)
print("Source revision:", manifest["git"]["revision"])
print("Working tree dirty:", manifest["git"]["dirty"])
print("Status:", manifest["status"])
print("Data kind:", manifest["data_kind"])

# Read actual run artifacts.
split_summary = pd.read_csv(run_dir / "split_summary.csv")
metrics = pd.read_csv(run_dir / "metrics.csv")

# Validate split sample counts.
expected_counts = {
    "train": 1000,
    "validation": 263,
    "test": 261,
    "unlabeled_dropped": 46,
}

assert {"split", "n_samples", "n_label_months"}.issubset(
    split_summary.columns
), f"Unexpected split schema: {split_summary.columns.tolist()}"

assert split_summary["split"].is_unique, (
    "Duplicate split names in split_summary.csv"
)

observed_counts = split_summary.set_index("split")["n_samples"].to_dict()
assert observed_counts == expected_counts, (
    f"Unexpected sample counts: {observed_counts}"
)

assert int(split_summary["n_samples"].sum()) == 1570

# Validate model/split coverage.
expected_models = {
    "B1_majority",
    "B1_persistence",
    "B4_logreg",
}
expected_splits = {"train", "validation", "test"}

assert len(metrics) == 9, (
    f"Expected 9 metric rows, got {len(metrics)}"
)
assert set(metrics["model"]) == expected_models
assert set(metrics["split"]) == expected_splits
assert not metrics.duplicated(["model", "split"]).any()

expected_pairs = {
    (model, split)
    for model in expected_models
    for split in expected_splits
}
observed_pairs = set(zip(metrics["model"], metrics["split"]))
assert observed_pairs == expected_pairs

# Validate metric values.
metric_columns = [
    "macro_f1",
    "macro_f1_six",
    "balanced_accuracy",
]

assert set(metric_columns).issubset(metrics.columns), (
    f"Missing metric columns: {metric_columns}"
)

for column in metric_columns:
    assert metrics[column].notna().all(), (
        f"Missing values in {column}"
    )
    assert metrics[column].between(0, 1).all(), (
        f"Out-of-range values in {column}"
    )

print("PASS: artifact consistency checks completed.")


display(split_summary)

display(
    metrics[
        [
            "model",
            "split",
            "n",
            "n_classes_present",
            "macro_f1",
            "macro_f1_six",
            "balanced_accuracy",
        ]
    ].sort_values(["split", "model"])
)

# Per-class evaluation metrics
per_class_path = run_dir / "per_class.csv"

if per_class_path.is_file():
    per_class = pd.read_csv(per_class_path)
    print("Per-class evaluation metrics")
    display(per_class)
else:
    print("per_class.csv is not available in this run.")

# Test confusion matrices for all baseline models
confusion_files = [
    "confusion_B1_majority_test.csv",
    "confusion_B1_persistence_test.csv",
    "confusion_B4_logreg_test.csv",
]

for filename in confusion_files:
    confusion_path = run_dir / filename
    print(f"\nTest confusion matrix: {filename}")

    if confusion_path.is_file():
        confusion_df = pd.read_csv(confusion_path)
        display(confusion_df)
    else:
        print(f"File not found: {confusion_path}")

## Handoff and limitations

Verified source revision: 696cbb61da2faebb75f06a701afee943fe0199c0.

The verified manifest records a clean working tree and successful completion of the table, split, baseline, and evaluation stages. The experiment is synthetic and covers only the three available baselines.

Outstanding work:
- Cross-check the synthetic generator rules, label timing, baseline fallback behavior, feature list, and preprocessing against the source implementation.
- Review per-class metrics and test confusion matrices, especially event classes with zero recall.
- Verify temporal boundaries, feature-time constraints, and the negative-control leakage test.
- Complete the remaining planned models: B2 Random Forest, B5 Gradient Boosting, and HeTCo.
- Evaluate on suitable real Reddit data when available.
- Cross-review evaluation outputs and document unresolved limitations.

The current evidence run uses synthetic data and the source revision recorded in its manifest. The results validate the current pipeline stages but do not establish real-world Reddit forecasting performance.

Before handoff, restart the kernel and run all cells. Save an executed local copy for review, then clear all outputs in the shared notebook before committing it. Do not claim that opening or editing the template alone completes an experiment.